# Preparing Raw Data (Preparando conjunto de dados)

This is necessary to try any subsection of this Notebook, so execute it!

---

Essa parte é necessária para executar qualquer subseção desse Notebook, portanto, é necessário executá-la sempre.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, PercentFormatter
import os
import optuna
from sklearn.decomposition import KernelPCA
from sklearn.metrics import mean_absolute_percentage_error

/home/guilherme/Repositories/data-driven-studies/droplet/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TESTED_MODES_COLLISION = np.linspace(1, 360, 30, dtype=int)
TESTED_MODES_SPREADING = np.linspace(1, 528, 30, dtype=int)
TRAIN_SPLIT = 0.8
# Kernels disponíveis para uso pelo SKLearn
SIMULATIONS = [
    'collision',
    'spreading'
]
KERNELS = {
    'poly': ['coef0', 'degree', 'gamma'],
    'rbf': ['gamma'],
    'sigmoid': ['coef0', 'gamma'],
}
RANGES = {
    'coef0': (-100., 100.),
    'degree': (1, 5),
    'gamma': (1e-5, 1.0),
}
NOW = pd.Timestamp.now().strftime('%Y-%m-%d_%H-%M-%S')
KERNEL_ID_TO_NAME = {
    'poly': 'polinomial',
    'rbf': 'gaussiano',
    'sigmoid': 'sigmoide',
}
display(NOW)
studies_folder = os.path.join('.studies', NOW)
os.makedirs(studies_folder, exist_ok=True)

'2024-11-21_01-08-39'

In [3]:
np.random.seed(42)

In [4]:
collision_file = np.load(".data/Time_series_colisao.npz")
display(collision_file)
spreading_file = np.load(".data/Time_series_espalhamento.npz")
display(spreading_file)

NpzFile '.data/Time_series_colisao.npz' with keys: TS, Re, We, B

NpzFile '.data/Time_series_espalhamento.npz' with keys: TS, Re, We, Fr, V0...

In [5]:
simulations_objs = {}
collision_timesteps_raw = collision_file['TS']
display(f"Dimensions of the collision simulations: {collision_timesteps_raw.shape}")
display(f"That is, {collision_timesteps_raw.shape[0]} simulations, with {collision_timesteps_raw.shape[1]} timesteps each, having {collision_timesteps_raw.shape[2]} components each.")
simulations_objs['collision'] = {
    'timesteps_raw': collision_timesteps_raw,
}
spreading_timesteps_raw = spreading_file['TS']
display(f"Dimensions of the spreading simulations: {spreading_timesteps_raw.shape}")
display(f"That is, {spreading_timesteps_raw.shape[0]} simulations, with {spreading_timesteps_raw.shape[1]} timesteps each, having {spreading_timesteps_raw.shape[2]} components each.")
simulations_objs['spreading'] = {
    'timesteps_raw': spreading_timesteps_raw,
}
del collision_file, spreading_file

'Dimensions of the collision simulations: (72, 1000, 5)'

'That is, 72 simulations, with 1000 timesteps each, having 5 components each.'

'Dimensions of the spreading simulations: (132, 1000, 4)'

'That is, 132 simulations, with 1000 timesteps each, having 4 components each.'

# Reordering data

In [6]:
collision_timesteps_components_last = np.transpose(collision_timesteps_raw, (1, 0, 2))
display(f"Shape after transpose is: {collision_timesteps_components_last.shape}")
# Flattening 2nd and 3rd dimensions
collision_timesteps_components_last = (
    collision_timesteps_components_last
    .reshape(collision_timesteps_components_last.shape[0], -1)
)
display(f"Shape after flattening is: {collision_timesteps_components_last.shape}")
simulations_objs['collision']['original_data'] = collision_timesteps_components_last

'Shape after transpose is: (1000, 72, 5)'

'Shape after flattening is: (1000, 360)'

In [7]:
spreading_timesteps_components_last = np.transpose(spreading_timesteps_raw, (1, 0, 2))
display(f"Shape after transpose is: {spreading_timesteps_components_last.shape}")
# Flattening 2nd and 3rd dimensions
spreading_timesteps_components_last = (
    spreading_timesteps_components_last
    .reshape(spreading_timesteps_components_last.shape[0], -1)
)
display(f"Shape after flattening is: {spreading_timesteps_components_last.shape}")
simulations_objs['spreading']['original_data'] = spreading_timesteps_components_last

'Shape after transpose is: (1000, 132, 4)'

'Shape after flattening is: (1000, 528)'

# Scaling data

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(simulations_objs['collision']['original_data'])
display(f"Asserting mean = 0 and std = 1 for every column")
assert np.allclose(scaled_data.mean(axis=0), 0)
assert np.allclose(scaled_data.std(axis=0), 1)
display("Success!")
simulations_objs['collision']['scaled_data'] = scaled_data
simulations_objs['collision']['scaler'] = scaler

'Asserting mean = 0 and std = 1 for every column'

'Success!'

In [9]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(simulations_objs['spreading']['original_data'])
display(f"Asserting mean = 0 and std = 1 for every column")
assert np.allclose(scaled_data.mean(axis=0), 0)
assert np.allclose(scaled_data.std(axis=0), 1)
display("Success!")
simulations_objs['spreading']['scaled_data'] = scaled_data
simulations_objs['spreading']['scaler'] = scaler

'Asserting mean = 0 and std = 1 for every column'

'Success!'

In [10]:
def objetive_function(trial: optuna.trial.Trial):
    studied_simulation = trial.study.user_attrs['studied_simulation']
    used_kernel = trial.study.user_attrs['used_kernel']
    studied_hyperparams = KERNELS[used_kernel]
    hyperparam_obj = {}
    if 'degree' in trial.study.user_attrs:
        hyperparam_obj['degree'] = trial.study.user_attrs['degree']
    if 'gamma' in studied_hyperparams:
        gamma = trial.suggest_float('gamma', RANGES['gamma'][0], RANGES['gamma'][1], log=True)
        hyperparam_obj['gamma'] = gamma
    if 'coef0' in studied_hyperparams:
        coef0 = trial.suggest_float('coef0', RANGES['coef0'][0], RANGES['coef0'][1])
        hyperparam_obj['coef0'] = coef0
    permutation_idx = np.random.permutation(simulations_objs[studied_simulation]['scaled_data'].shape[0])
    number_of_timesteps = simulations_objs[studied_simulation]['scaled_data'].shape[0]
    number_of_timesteps_train = int(number_of_timesteps * TRAIN_SPLIT)
    training_idx = permutation_idx[:number_of_timesteps_train]
    testing_idx = permutation_idx[number_of_timesteps_train:]
    training_data = simulations_objs[studied_simulation]['original_data'][training_idx]
    training_data_scaled = simulations_objs[studied_simulation]['scaled_data'][training_idx]
    testing_data = simulations_objs[studied_simulation]['original_data'][testing_idx]
    testing_data_scaled = simulations_objs[studied_simulation]['scaled_data'][testing_idx]
    scaler = simulations_objs[studied_simulation]['scaler']
    tested_modes = TESTED_MODES_COLLISION if studied_simulation == 'collision' else TESTED_MODES_SPREADING
    try:
        mape_errors_test = np.zeros(len(tested_modes))
        mape_errors_train = np.zeros(len(tested_modes))
        max_errors = np.zeros(len(tested_modes))
        min_errors = np.zeros(len(tested_modes))
        for i, mode in enumerate(tested_modes):
            kpca = KernelPCA(n_components=mode, kernel=used_kernel, fit_inverse_transform=True, **hyperparam_obj)
            kpca.fit(training_data_scaled)
            reconstructed_training_scaled = kpca.inverse_transform(kpca.transform(training_data_scaled))
            reconstructed_training = scaler.inverse_transform(reconstructed_training_scaled)
            mape_per_row_training = mean_absolute_percentage_error(training_data, reconstructed_training, multioutput='raw_values') * 100
            reconstructed_testing_scaled = kpca.inverse_transform(kpca.transform(testing_data_scaled))
            reconstructed_testing = scaler.inverse_transform(reconstructed_testing_scaled)
            mape_per_row_testing = mean_absolute_percentage_error(testing_data, reconstructed_testing, multioutput='raw_values') * 100
            mape_errors_train[i] = np.mean(mape_per_row_training)
            mape_errors_test[i] = np.mean(mape_per_row_testing)
            all_errors = np.append(mape_per_row_training, mape_per_row_testing)
            max_errors[i] = np.max(all_errors)
            min_errors[i] = np.min(all_errors)
        fig, ax = plt.subplots()
        ax.plot(tested_modes, mape_errors_test, '-', label='Erro médio de teste')
        ax.plot(tested_modes, mape_errors_train, '--', label='Erro médio de treino', alpha=0.5)
        ax.plot(tested_modes, max_errors, ':r', label='Erro máximo', alpha=(1/3))
        ax.plot(tested_modes, min_errors, ':g', label='Erro mínimo', alpha=(1/3))
        ax.set_title(f"Erro médio absoluto percentual para {studied_simulation} com kernel {KERNEL_ID_TO_NAME[used_kernel]}")
        ax.set_xlabel('N. de componentes')
        ax.set_ylabel('Erro percentual')
        ax.yaxis.set_major_locator(MaxNLocator())
        ax.yaxis.set_major_formatter(PercentFormatter())
        ax.legend()
        hyperparams_str = '\n'.join([f"{key}: {value}" for key, value in hyperparam_obj.items()])
        ax.annotate(hyperparams_str, xy=(0.05, 0.8), xycoords='axes fraction', fontsize=8)
        trial_folder = os.path.join(studies_folder, trial.study.study_name)
        os.makedirs(trial_folder, exist_ok=True)
        fig.savefig(os.path.join(trial_folder, f"error_plot_{trial.number}.png"))
        # ignore plot
        plt.close(fig)
        # save .npz
        np.savez_compressed(
            os.path.join(trial_folder, f"data_{trial.number}.npz"),
            mape_errors_test=mape_errors_test,
            mape_errors_train=mape_errors_train,
            max_errors=max_errors,
            min_errors=min_errors,
            tested_modes=tested_modes,
            reconstructed_training=reconstructed_training,
            reconstructed_testing=reconstructed_testing,
            training_data=training_data,
            testing_data=testing_data,
        )
        minimal_max_error = np.min(max_errors)
        mean_max_error_decay = np.mean(np.diff(max_errors))
    except Exception as e:
        trial.set_user_attr('exception', e.__class__.__name__)
        print("Error in trial", e)
        minimal_max_error = float('inf')
        mean_max_error_decay = float('inf')
    return minimal_max_error, mean_max_error_decay

In [12]:
from itertools import product

os.makedirs('.optunadb', exist_ok=True)
for simulation, kernel in product(SIMULATIONS, KERNELS.keys()):
    if kernel == 'poly':
        for degree in range(1, 5):
            study = optuna.create_study(
                directions=['minimize', 'minimize'],
                study_name=f"{simulation}_{kernel}_degree_{degree}",
                storage=f"sqlite:///.optunadb/{simulation}_{kernel}_degree_{degree}.db",
                load_if_exists=True,
            )
            study.set_user_attr('degree', degree)
            study.set_user_attr('studied_simulation', simulation)
            study.set_user_attr('used_kernel', kernel)
            remaining_to_50 = 50 - len(study.trials)
            study.optimize(objetive_function, n_trials=remaining_to_50, show_progress_bar=True)
    else:
        study = optuna.create_study(
            directions=['minimize', 'minimize'],
            study_name=f"{simulation}_{kernel}",
            storage=f"sqlite:///.optunadb/{simulation}_{kernel}.db",
            load_if_exists=True,
        )
        study.set_user_attr('studied_simulation', simulation)
        study.set_user_attr('used_kernel', kernel)
        remaining_to_50 = 50 - len(study.trials)
        study.optimize(objetive_function, n_trials=remaining_to_50, show_progress_bar=True)

[I 2024-11-21 01:29:00,202] Using an existing study with name 'collision_poly_degree_1' instead of creating a new one.
[I 2024-11-21 01:29:00,254] Using an existing study with name 'collision_poly_degree_2' instead of creating a new one.
[I 2024-11-21 01:29:00,300] Using an existing study with name 'collision_poly_degree_3' instead of creating a new one.
[I 2024-11-21 01:29:00,347] Using an existing study with name 'collision_poly_degree_4' instead of creating a new one.
[I 2024-11-21 01:29:00,541] A new study created in RDB with name: collision_rbf
  2%|▏         | 1/50 [00:16<13:32, 16.59s/it]

[I 2024-11-21 01:29:17,147] Trial 0 finished with values: [2314.955167471243, 5.247024514286232] and parameters: {'gamma': 0.005853885068568264}.


  4%|▍         | 2/50 [00:29<11:46, 14.71s/it]

[I 2024-11-21 01:29:30,539] Trial 1 finished with values: [4136.63464296762, 0.1027873081220605] and parameters: {'gamma': 0.0027898676843345622}.


  6%|▌         | 3/50 [00:44<11:19, 14.45s/it]

...

[I 2024-11-21 02:24:50,513] Trial 46 finished with values: [15370334.598645737, 0.0] and parameters: {'gamma': 0.006195870278642749, 'coef0': 53.554909787117595}.
Error in trial Matrix is singular.
[I 2024-11-21 02:24:50,655] Trial 47 finished with values: [inf, inf] and parameters: {'gamma': 0.32763775368449505, 'coef0': -79.55396809774322}.


 98%|█████████▊| 49/50 [05:13<00:07,  7.48s/it]

[I 2024-11-21 02:25:05,240] Trial 48 finished with values: [14221946.09069867, 0.0] and parameters: {'gamma': 0.0003954321119039954, 'coef0': 34.28727649708529}.


100%|██████████| 50/50 [05:29<00:00,  6.59s/it]

[I 2024-11-21 02:25:21,269] Trial 49 finished with values: [13379606.620171407, 0.0] and parameters: {'gamma': 1.4647992466523243e-05, 'coef0': 11.250546879236438}.
